# 1️⃣ Install Kaggle and upload API key

In [ ]:
!pip install -q kaggle
from google.colab import files

# Upload your kaggle.json API key
files.upload()


Saving kaggle.json to kaggle (2).json


{'kaggle (2).json': b'{"username":"dobserine","key":"c2e4dbbb48001a136dbc27ad4ad9dfc6"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


# 2️⃣ Download competition files

In [ ]:
!kaggle competitions download -c micro-club-pinktober-brain-tumor-classification


micro-club-pinktober-brain-tumor-classification.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -q micro-club-pinktober-brain-tumor-classification.zip -d ./data


replace ./data/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


# 3️⃣ Load data into Pandas

In [ ]:
import pandas as pd

train_df = pd.read_csv('./data/train.csv')
test_df = pd.read_csv('./data/test.csv')
sample_submission = pd.read_csv('./data/sample_submission.csv')

print("✅ Train shape:", train_df.shape)
print("✅ Test shape:", test_df.shape)
print("✅ Sample submission shape:", sample_submission.shape)
train_df.head()


✅ Train shape: (7000, 20)
✅ Test shape: (3000, 19)
✅ Sample submission shape: (3000, 2)


,tumor_type,size,location,edema,necrosis,enhancement,shape,margins,calcification,cystic_components,hemorrhage,ki67_index,mitotic_count,age,gender,symptoms_duration,neurological_deficit,kps_score,cancer_stage,id
0,pituitary,khlat_3lik,frontal,1,0,none,irregular,poorly_defined,1,0,0,100.0,19,65,female,233,0,90,IV,0
1,glioma,normal_brk,frontal,0,0,none,irregular,well_defined,0,1,0,40.0,13,84,amira,233,1,60,IV,1
2,metastatic,normal_brk,occipital,1,0,mild,irregular,well_defined,1,0,0,95.0,2,79,wa7ch,19,1,60,IV,2
3,meningioma,normal_brk,frontal,1,1,none,irregular,poorly_defined,1,0,0,100.0,13,71,wa7ch,157,0,80,IV,3
4,meningioma,normal_brk,brainstem,0,1,ring,irregular,well_defined,0,0,0,25.0,18,31,amira,207,1,90,IV,4


In [ ]:
!pip install -q catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.4 MB/s eta 0:00:00


# 📦 Libraries


In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
import random, os, warnings

warnings.filterwarnings("ignore")

# 🎯 Seed for Reproducibility

In [30]:
RANDOM_SEED = 123
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# 1️⃣ Load Dataset

In [31]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
test_identifiers = df_test['id'].copy()

# Categorical columns to one-hot encode
cat_features = ['tumor_type', 'size', 'location', 'enhancement', 'shape', 'margins']

# 2️⃣ Data Preprocessing

In [32]:
X = pd.get_dummies(df_train.drop(['cancer_stage', 'id', 'gender'], axis=1), columns=cat_features)
y = df_train['cancer_stage']

# Encode target if necessary
if y.dtype == 'object':
    label_enc = LabelEncoder()
    y = label_enc.fit_transform(y)

# Feature scaling
scaler_model = StandardScaler()
X_scaled = scaler_model.fit_transform(X)

# Select important features (from previous analysis)
selected_features = [
    'edema', 'necrosis', 'calcification', 'cystic_components', 'hemorrhage', 'ki67_index',
    'mitotic_count', 'age', 'kps_score', 'tumor_type_glioma', 'tumor_type_meningioma',
    'tumor_type_metastatic', 'tumor_type_pituitary', 'size_kbira', 'size_khlat_3lik',
    'size_sghir_bzef', 'size_sghira', 'location_brainstem', 'location_cerebellum',
    'location_frontal', 'location_occipital', 'location_temporal', 'enhancement_mild',
    'enhancement_none', 'enhancement_ring', 'enhancement_strong', 'shape_regular',
    'margins_poorly_defined', 'margins_well_defined'
]

feature_idx = [X.columns.get_loc(col) for col in selected_features]
X_selected = X_scaled[:, feature_idx]

# Train/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_selected, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# 3️⃣ Model Definition

In [33]:
lr_model = LogisticRegression(C=10, solver='saga', max_iter=5000, multi_class='multinomial', random_state=RANDOM_SEED)
rf_model = RandomForestClassifier(n_estimators=300, max_depth=10, n_jobs=-1, random_state=RANDOM_SEED)
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(np.unique(y_tr)),
    random_state=RANDOM_SEED,
    eval_metric='mlogloss'
)

meta_xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(np.unique(y_tr)),
    random_state=RANDOM_SEED,
    eval_metric='mlogloss'
)

stacking_ensemble = StackingClassifier(
    estimators=[('lr', lr_model), ('rf', rf_model), ('xgb', xgb_model)],
    final_estimator=meta_xgb,
    passthrough=True,
    n_jobs=-1
)


# 4️⃣ Train Model

In [34]:
stacking_ensemble.fit(X_tr, y_tr)

StackingClassifier(estimators=[('lr',
                                LogisticRegression(C=10, max_iter=5000,
                                                   multi_class='multinomial',
                                                   random_state=123,
                                                   solver='saga')),
                               ('rf',
                                RandomForestClassifier(max_depth=10,
                                                       n_estimators=300,
                                                       n_jobs=-1,
                                                       random_state=123)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0...
                                                 gamma=None, grow_policy=None,
                                                 importance_type=None,
                                                 interaction_constraints=None,
                                                 learning_rate=0.05,
                                                 max_bin=None,
                                                 max_cat_threshold=None,
                                                 max_cat_to_onehot=None,
                                                 max_delta_step=None,
                                                 max_depth=4, max_leaves=None,
                                                 min_child_weight=None,
                                                 missing=nan,
                                                 monotone_constraints=None,
                                                 multi_strategy=None,
                                                 n_estimators=300, n_jobs=None,
                                                 num_class=4, ...),
                   n_jobs=-1, passthrough=True)

# 5️⃣ Evaluate on Validation

In [36]:
val_pred = stacking_ensemble.predict(X_val)
val_f1_score = f1_score(y_val, val_pred, average='weighted')
print(f"Validation Weighted F1 Score: {val_f1_score:.4f}")
print("\nClassification Report:\n", classification_report(y_val, val_pred))


Validation Weighted F1 Score: 0.8442

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        50
           1       0.84      0.43      0.57        96
           2       0.83      0.84      0.83       307
           3       0.88      0.97      0.92       947

    accuracy                           0.87      1400
   macro avg       0.64      0.56      0.58      1400
weighted avg       0.83      0.87      0.84      1400



# 6️⃣ Prepare Test Data & Predict

In [37]:
X_test = pd.get_dummies(df_test.drop(['id', 'gender'], axis=1), columns=cat_features)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
X_test_scaled = scaler_model.transform(X_test)
X_test_selected = X_test_scaled[:, feature_idx]

test_pred = stacking_ensemble.predict(X_test_selected)

# Decode labels if LabelEncoder was applied
if 'label_enc' in locals():
    test_pred = label_enc.inverse_transform(test_pred)

# 7️⃣ Save Submission

In [38]:
submission_df = pd.DataFrame({'id': test_identifiers, 'cancer_stage': test_pred})
submission_df.to_csv('submission.csv', index=False)
print("✅ submission.csv generated successfully!")


✅ submission.csv generated successfully!
